# Stanford RNA 3D Folding Part 2 - Complete Notebook

Self-contained notebook for training and generating 3D structure predictions for RNA sequences.

**Competition**: [Stanford RNA 3D Folding Part 2](https://www.kaggle.com/competitions/stanford-rna-3d-folding-2)

**Task**: Predict 3D C1' atom coordinates for RNA molecules from sequence.

**Metric**: TM-score (best of 5 predictions per target)

In [ ]:
import os
import sys
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from einops import rearrange
from tqdm import tqdm

# Check environment
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Configuration

In [ ]:
# Auto-detect environment: Kaggle vs local
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    INPUT_DIR = '/kaggle/input/stanford-rna-3d-folding-2'
    MODEL_DIR = '/kaggle/input/rna-fold-model'
    OUTPUT_DIR = '/kaggle/working'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
    INPUT_DIR = os.path.join(PROJECT_ROOT, 'data')
    MODEL_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
    OUTPUT_DIR = PROJECT_ROOT

# Training save_dir must be writable — on Kaggle, /kaggle/input is read-only
SAVE_DIR = '/kaggle/working/checkpoints' if IS_KAGGLE else MODEL_DIR

CONFIG = {
    'data': {
        'train_dir': os.path.join(INPUT_DIR, 'train') if not IS_KAGGLE else os.path.join(INPUT_DIR),
        'test_csv': os.path.join(INPUT_DIR, 'test_sequences.csv'),
        'max_seq_len': 512,
        'num_workers': 4,
    },
    'model': {
        'd_model': 256,
        'n_heads': 8,
        'n_layers': 8,
        'd_ff': 1024,
        'dropout': 0.1,
        'num_predictions': 5,
    },
    'training': {
        'batch_size': 4,
        'learning_rate': 3.0e-4,
        'weight_decay': 1.0e-4,
        'num_epochs': 100,
        'warmup_steps': 1000,
        'grad_clip': 1.0,
        'seed': 42,
        'save_dir': SAVE_DIR,
    },
}

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Input dir: {INPUT_DIR}')
print(f'Model dir (weights): {MODEL_DIR}')
print(f'Save dir (training): {SAVE_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

## 2. Dataset

Nucleotide vocabulary, sequence encoding, PDB/mmCIF parsing, and dataset classes.

In [ ]:
# Nucleotide vocabulary: A, C, G, U + padding + unknown
NUC_VOCAB = {'<pad>': 0, 'A': 1, 'C': 2, 'G': 3, 'U': 4, '<unk>': 5}
VOCAB_SIZE = len(NUC_VOCAB)


def encode_sequence(seq):
    """Encode an RNA sequence string to integer tokens."""
    return [NUC_VOCAB.get(c.upper(), NUC_VOCAB['<unk>']) for c in seq]


def parse_pdb_c1_coords(pdb_path):
    """Extract C1' atom coordinates from a PDB file."""
    coords = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                atom_name = line[12:16].strip()
                if atom_name == "C1'":
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    coords.append([x, y, z])
    return np.array(coords, dtype=np.float32) if coords else np.zeros((0, 3), dtype=np.float32)


def parse_mmcif_c1_coords(cif_path):
    """Extract C1' atom coordinates from an mmCIF file."""
    coords = []
    in_atom_site = False
    col_names = []
    x_idx = y_idx = z_idx = atom_idx = -1

    with open(cif_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('_atom_site.'):
                in_atom_site = True
                col_name = line.split('.')[1].split()[0]
                col_names.append(col_name)
                if col_name == 'Cartn_x':
                    x_idx = len(col_names) - 1
                elif col_name == 'Cartn_y':
                    y_idx = len(col_names) - 1
                elif col_name == 'Cartn_z':
                    z_idx = len(col_names) - 1
                elif col_name == 'label_atom_id':
                    atom_idx = len(col_names) - 1
            elif in_atom_site and (line.startswith('ATOM') or line.startswith('HETATM')):
                parts = line.split()
                if len(parts) > max(x_idx, y_idx, z_idx, atom_idx):
                    atom_name = parts[atom_idx].strip("'\"")
                    if atom_name == "C1'":
                        x = float(parts[x_idx])
                        y = float(parts[y_idx])
                        z = float(parts[z_idx])
                        coords.append([x, y, z])
            elif in_atom_site and not line.startswith(('ATOM', 'HETATM', '_', '#', 'loop_')):
                if line and not line.startswith(';'):
                    in_atom_site = False

    return np.array(coords, dtype=np.float32) if coords else np.zeros((0, 3), dtype=np.float32)


class RNATrainDataset(Dataset):
    """Training dataset: RNA sequences paired with 3D C1' coordinates."""

    def __init__(self, train_dir, max_seq_len=512):
        self.max_seq_len = max_seq_len
        self.train_dir = train_dir

        seq_csv = os.path.join(train_dir, 'sequences.csv')
        if os.path.exists(seq_csv):
            self.df = pd.read_csv(seq_csv)
        else:
            self.df = self._build_from_structures(train_dir)

        self.df = self.df[self.df['sequence'].str.len() <= max_seq_len].reset_index(drop=True)

    def _build_from_structures(self, train_dir):
        struct_dir = os.path.join(train_dir, 'structures')
        records = []
        if os.path.exists(struct_dir):
            for fname in os.listdir(struct_dir):
                if fname.endswith(('.pdb', '.cif')):
                    target_id = os.path.splitext(fname)[0]
                    fpath = os.path.join(struct_dir, fname)
                    seq = self._extract_sequence_from_pdb(fpath) if fname.endswith('.pdb') else ''
                    if seq:
                        records.append({'target_id': target_id, 'sequence': seq, 'structure_file': fname})
        return pd.DataFrame(records) if records else pd.DataFrame(columns=['target_id', 'sequence', 'structure_file'])

    @staticmethod
    def _extract_sequence_from_pdb(pdb_path):
        res_map = {'A': 'A', 'C': 'C', 'G': 'G', 'U': 'U',
                   'DA': 'A', 'DC': 'C', 'DG': 'G', 'DT': 'U',
                   'ADE': 'A', 'CYT': 'C', 'GUA': 'G', 'URA': 'U'}
        residues = []
        seen = set()
        with open(pdb_path, 'r') as f:
            for line in f:
                if line.startswith(('ATOM', 'HETATM')):
                    atom_name = line[12:16].strip()
                    if atom_name == "C1'":
                        resname = line[17:20].strip()
                        resseq = line[22:26].strip()
                        chain = line[21]
                        key = (chain, resseq)
                        if key not in seen:
                            seen.add(key)
                            nuc = res_map.get(resname, '')
                            if nuc:
                                residues.append((int(resseq), nuc))
        residues.sort()
        return ''.join(r[1] for r in residues)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row['sequence']
        target_id = row['target_id']

        tokens = encode_sequence(seq)
        seq_len = len(tokens)
        padded = tokens + [0] * (self.max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (self.max_seq_len - seq_len)

        coords = np.zeros((self.max_seq_len, 3), dtype=np.float32)
        struct_dir = os.path.join(self.train_dir, 'structures')
        for ext in ['.pdb', '.cif']:
            struct_path = os.path.join(struct_dir, target_id + ext)
            if os.path.exists(struct_path):
                if ext == '.pdb':
                    raw_coords = parse_pdb_c1_coords(struct_path)
                else:
                    raw_coords = parse_mmcif_c1_coords(struct_path)
                n = min(len(raw_coords), seq_len, self.max_seq_len)
                coords[:n] = raw_coords[:n]
                break

        return {
            'tokens': torch.tensor(padded, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
            'coords': torch.tensor(coords, dtype=torch.float32),
            'seq_len': seq_len,
            'target_id': target_id,
        }


class RNATestDataset(Dataset):
    """Test dataset: RNA sequences only (no labels)."""

    def __init__(self, csv_path, max_seq_len=512):
        self.max_seq_len = max_seq_len
        self.df = pd.read_csv(csv_path)
        assert 'target_id' in self.df.columns
        assert 'sequence' in self.df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row['sequence']
        target_id = row['target_id']

        tokens = encode_sequence(seq)
        seq_len = len(tokens)
        if seq_len > self.max_seq_len:
            tokens = tokens[:self.max_seq_len]
            seq_len = self.max_seq_len

        padded = tokens + [0] * (self.max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (self.max_seq_len - seq_len)

        result = {
            'tokens': torch.tensor(padded, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
            'seq_len': seq_len,
            'target_id': target_id,
            'sequence': seq,
        }
        if 'description' in self.df.columns:
            result['description'] = row.get('description', '')
        if 'all_sequences' in self.df.columns:
            result['all_sequences'] = row.get('all_sequences', '')
        return result


def get_train_dataloader(train_dir, max_seq_len=512, batch_size=4, num_workers=4):
    dataset = RNATrainDataset(train_dir, max_seq_len)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True,
                      num_workers=num_workers, pin_memory=True)


print(f'Dataset classes defined. Vocab size: {VOCAB_SIZE}')

## 3. Model

Transformer-based architecture for predicting 3D C1' atom coordinates from RNA sequences.

Architecture:
1. Sequence Embedding: nucleotide tokens + positional encoding
2. Pairwise Representation: outer product of single representations
3. Transformer Encoder: self-attention with pairwise bias
4. Structure Module: predicts 3D coordinates via MLP heads
5. Multi-prediction Head: outputs 5 coordinate sets for ensemble scoring

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class PairwiseModule(nn.Module):
    """Computes pairwise representations via outer product, then refines with MLP."""

    def __init__(self, d_model, d_pair=64):
        super().__init__()
        self.proj_left = nn.Linear(d_model, d_pair)
        self.proj_right = nn.Linear(d_model, d_pair)
        self.pair_norm = nn.LayerNorm(d_pair)
        self.pair_mlp = nn.Sequential(
            nn.Linear(d_pair, d_pair * 2), nn.GELU(), nn.Linear(d_pair * 2, d_pair)
        )
        self.pair_to_bias = nn.Linear(d_pair, 1)

    def forward(self, single_repr, mask):
        left = self.proj_left(single_repr)
        right = self.proj_right(single_repr)
        pair_repr = torch.einsum('bid,bjd->bijd', left, right)
        pair_repr = self.pair_norm(pair_repr)
        pair_repr = pair_repr + self.pair_mlp(pair_repr)
        pair_mask = mask.unsqueeze(-1) * mask.unsqueeze(-2)
        pair_repr = pair_repr * pair_mask.unsqueeze(-1)
        attn_bias = self.pair_to_bias(pair_repr).squeeze(-1)
        return pair_repr, attn_bias


class StructureAwareAttention(nn.Module):
    """Multi-head self-attention with pairwise bias."""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, attn_bias=None):
        B, L, D = x.shape
        q = rearrange(self.q_proj(x), 'b l (h d) -> b h l d', h=self.n_heads)
        k = rearrange(self.k_proj(x), 'b l (h d) -> b h l d', h=self.n_heads)
        v = rearrange(self.v_proj(x), 'b l (h d) -> b h l d', h=self.n_heads)
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if attn_bias is not None:
            attn = attn + attn_bias.unsqueeze(1)
        mask_2d = mask.unsqueeze(1).unsqueeze(2)
        attn = attn.masked_fill(~mask_2d, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h l d -> b l (h d)')
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = StructureAwareAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, mask, attn_bias=None):
        x = x + self.attn(self.norm1(x), mask, attn_bias)
        x = x + self.ffn(self.norm2(x))
        return x


class StructureModule(nn.Module):
    """Predicts 3D coordinates and per-residue confidence."""

    def __init__(self, d_model, num_predictions=5):
        super().__init__()
        self.num_predictions = num_predictions
        self.coord_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model), nn.GELU(),
                nn.Linear(d_model, d_model // 2), nn.GELU(),
                nn.Linear(d_model // 2, 3),
            ) for _ in range(num_predictions)
        ])
        self.confidence_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, num_predictions), nn.Sigmoid(),
        )

    def forward(self, single_repr):
        coords = torch.stack([head(single_repr) for head in self.coord_heads], dim=2)
        confidence = self.confidence_head(single_repr)
        return {'coords': coords, 'confidence': confidence}


class RNAFoldModel(nn.Module):
    """End-to-end RNA 3D structure prediction model."""

    def __init__(self, d_model=256, n_heads=8, n_layers=8, d_ff=1024,
                 dropout=0.1, num_predictions=5, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.num_predictions = num_predictions
        self.token_emb = nn.Embedding(VOCAB_SIZE, d_model, padding_idx=0)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=max_seq_len + 100)
        self.pairwise = PairwiseModule(d_model, d_pair=64)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.structure_module = StructureModule(d_model, num_predictions)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, tokens, mask):
        x = self.token_emb(tokens)
        x = self.pos_enc(x)
        _, attn_bias = self.pairwise(x, mask)
        for layer in self.layers:
            x = x * mask.unsqueeze(-1).float()
            x = layer(x, mask, attn_bias)
        return self.structure_module(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print('Model defined successfully.')

## 4. Loss Functions and Metrics

FAPE loss, distance matrix loss, TM-score, and combined training loss.

In [ ]:
def compute_distance_matrix(coords):
    """Compute pairwise distance matrix. coords: (B, L, 3) -> (B, L, L)"""
    diff = coords.unsqueeze(2) - coords.unsqueeze(1)
    return torch.sqrt((diff ** 2).sum(-1) + 1e-8)


def distance_matrix_loss(pred_coords, true_coords, mask):
    """L1 loss on pairwise distance matrices (rotationally invariant)."""
    pred_dist = compute_distance_matrix(pred_coords)
    true_dist = compute_distance_matrix(true_coords)
    pair_mask = (mask.unsqueeze(-1) * mask.unsqueeze(-2)).float()
    loss = (pred_dist - true_dist).abs() * pair_mask
    return loss.sum() / (pair_mask.sum() + 1e-8)


def fape_loss(pred_coords, true_coords, mask, clamp_distance=10.0):
    """Frame Aligned Point Error (simplified) with clamping."""
    mask_float = mask.float().unsqueeze(-1)
    num_valid = mask_float.sum(dim=1, keepdim=True).clamp(min=1)

    pred_center = (pred_coords * mask_float).sum(dim=1, keepdim=True) / num_valid
    true_center = (true_coords * mask_float).sum(dim=1, keepdim=True) / num_valid

    pred_centered = pred_coords - pred_center
    true_centered = true_coords - true_center

    diff = pred_centered - true_centered
    per_residue_dist = torch.sqrt((diff ** 2).sum(-1) + 1e-8)
    per_residue_dist = torch.clamp(per_residue_dist, max=clamp_distance)

    loss = (per_residue_dist * mask.float()).sum() / (mask.float().sum() + 1e-8)
    return loss


def kabsch_align(pred, true):
    """Align pred to true using Kabsch algorithm (optimal rotation)."""
    pred_center = pred.mean(axis=0)
    true_center = true.mean(axis=0)
    pred_c = pred - pred_center
    true_c = true - true_center
    H = pred_c.T @ true_c
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1, 1, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    return pred_c @ R.T + true_center


def compute_tm_score(pred_coords, true_coords):
    """Compute TM-score between predicted and true structures."""
    L = len(true_coords)
    if L == 0:
        return 0.0
    d0 = max(0.6 * (L - 0.5) ** (1.0 / 3.0) - 2.5, 0.5)
    aligned = kabsch_align(pred_coords, true_coords)
    dists = np.sqrt(((aligned - true_coords) ** 2).sum(axis=1))
    return float((1.0 / (1.0 + (dists / d0) ** 2)).sum() / L)


def best_of_n_tm_score(pred_coords_list, true_coords):
    """Compute best-of-N TM-score (competition metric)."""
    return max(compute_tm_score(pred, true_coords) for pred in pred_coords_list)


class CombinedLoss(nn.Module):
    """Combined training loss: FAPE + distance matrix + confidence."""

    def __init__(self, fape_weight=1.0, dist_weight=0.5, confidence_weight=0.1):
        super().__init__()
        self.fape_weight = fape_weight
        self.dist_weight = dist_weight
        self.confidence_weight = confidence_weight

    def forward(self, pred, true_coords, mask):
        num_preds = pred['coords'].shape[2]
        total_fape = 0.0
        total_dist = 0.0
        per_pred_losses = []

        for i in range(num_preds):
            pred_i = pred['coords'][:, :, i, :]
            f = fape_loss(pred_i, true_coords, mask)
            d = distance_matrix_loss(pred_i, true_coords, mask)
            total_fape += f
            total_dist += d
            per_pred_losses.append((f + d).detach())

        total_fape /= num_preds
        total_dist /= num_preds

        confidence_loss = torch.tensor(0.0, device=true_coords.device)
        if self.confidence_weight > 0:
            per_pred_losses_t = torch.stack(per_pred_losses)
            with torch.no_grad():
                quality = 1.0 / (1.0 + per_pred_losses_t)
                quality = quality / (quality.max() + 1e-8)
                quality_target = quality.unsqueeze(0).unsqueeze(0).expand_as(pred['confidence'])
            confidence_loss = F.mse_loss(
                pred['confidence'] * mask.unsqueeze(-1).float(),
                quality_target * mask.unsqueeze(-1).float(),
            )

        total_loss = (
            self.fape_weight * total_fape
            + self.dist_weight * total_dist
            + self.confidence_weight * confidence_loss
        )

        return {
            'loss': total_loss,
            'fape_loss': total_fape,
            'dist_loss': total_dist,
            'confidence_loss': confidence_loss,
        }


print('Loss functions and metrics defined.')

## 5. Training Pipeline

Mixed precision training with warmup + cosine decay, gradient clipping, TM-score validation, and checkpointing.

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_lr_scheduler(optimizer, warmup_steps, total_steps):
    """Linear warmup then cosine decay."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion,
                    scaler, device, grad_clip, epoch):
    model.train()
    total_loss = 0
    total_fape = 0
    total_dist = 0
    n_batches = 0

    pbar = tqdm(dataloader, desc=f'Epoch {epoch}')
    for batch in pbar:
        tokens = batch['tokens'].to(device)
        mask = batch['mask'].to(device)
        coords = batch['coords'].to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda', enabled=scaler.is_enabled()):
            pred = model(tokens, mask)
            loss_dict = criterion(pred, coords, mask)
            loss = loss_dict['loss']

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        total_fape += loss_dict['fape_loss'].item()
        total_dist += loss_dict['dist_loss'].item()
        n_batches += 1

        pbar.set_postfix({
            'loss': f'{total_loss / n_batches:.4f}',
            'fape': f'{total_fape / n_batches:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}',
        })

    return {
        'loss': total_loss / max(n_batches, 1),
        'fape_loss': total_fape / max(n_batches, 1),
        'dist_loss': total_dist / max(n_batches, 1),
    }


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_tm = 0
    n_batches = 0
    n_tm = 0

    for batch in tqdm(dataloader, desc='Validating'):
        tokens = batch['tokens'].to(device)
        mask = batch['mask'].to(device)
        coords = batch['coords'].to(device)

        pred = model(tokens, mask)
        loss_dict = criterion(pred, coords, mask)
        total_loss += loss_dict['loss'].item()
        n_batches += 1

        pred_coords = pred['coords'].cpu().numpy()
        true_coords = coords.cpu().numpy()
        masks = mask.cpu().numpy()

        for b in range(pred_coords.shape[0]):
            seq_len = masks[b].sum().astype(int)
            if seq_len < 3:
                continue
            true_c = true_coords[b, :seq_len]
            if np.abs(true_c).sum() < 1e-6:
                continue
            pred_list = [pred_coords[b, :seq_len, i] for i in range(pred_coords.shape[2])]
            tm = max(compute_tm_score(p, true_c) for p in pred_list)
            total_tm += tm
            n_tm += 1

    return {
        'loss': total_loss / max(n_batches, 1),
        'tm_score': total_tm / max(n_tm, 1),
    }


print('Training pipeline defined.')

## 6. Train the Model

Run training with the configuration above. Skip this section if loading pretrained weights.

In [ ]:
# Set to True to run training, False to skip to inference
RUN_TRAINING = False

if RUN_TRAINING:
    cfg = CONFIG
    set_seed(cfg['training']['seed'])
    os.makedirs(cfg['training']['save_dir'], exist_ok=True)

    # Data
    train_loader = get_train_dataloader(
        cfg['data']['train_dir'],
        max_seq_len=cfg['data']['max_seq_len'],
        batch_size=cfg['training']['batch_size'],
        num_workers=cfg['data']['num_workers'],
    )

    full_dataset = train_loader.dataset
    n_val = max(1, len(full_dataset) // 10)
    n_train = len(full_dataset) - n_val
    train_subset, val_subset = torch.utils.data.random_split(full_dataset, [n_train, n_val])

    train_loader = DataLoader(
        train_subset, batch_size=cfg['training']['batch_size'],
        shuffle=True, num_workers=cfg['data']['num_workers'], pin_memory=True,
    )
    val_loader = DataLoader(
        val_subset, batch_size=cfg['training']['batch_size'],
        shuffle=False, num_workers=0,
    )

    # Model
    model = RNAFoldModel(
        d_model=cfg['model']['d_model'],
        n_heads=cfg['model']['n_heads'],
        n_layers=cfg['model']['n_layers'],
        d_ff=cfg['model']['d_ff'],
        dropout=cfg['model']['dropout'],
        num_predictions=cfg['model']['num_predictions'],
        max_seq_len=cfg['data']['max_seq_len'],
    ).to(device)
    print(f'Model parameters: {model.count_parameters():,}')

    # Optimizer + scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg['training']['learning_rate'],
        weight_decay=cfg['training']['weight_decay'],
    )
    total_steps = cfg['training']['num_epochs'] * len(train_loader)
    scheduler = get_lr_scheduler(optimizer, cfg['training']['warmup_steps'], total_steps)
    criterion = CombinedLoss()
    scaler = GradScaler(enabled=device.type == 'cuda')

    best_tm = 0.0

    for epoch in range(cfg['training']['num_epochs']):
        t0 = time.time()
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            scaler, device, cfg['training']['grad_clip'], epoch,
        )
        val_metrics = validate(model, val_loader, criterion, device)
        elapsed = time.time() - t0

        print(
            f'Epoch {epoch}: '
            f'train_loss={train_metrics["loss"]:.4f} '
            f'val_loss={val_metrics["loss"]:.4f} '
            f'val_tm={val_metrics["tm_score"]:.4f} '
            f'time={elapsed:.1f}s'
        )

        ckpt = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': epoch,
            'best_tm': best_tm,
            'config': cfg,
        }

        if val_metrics['tm_score'] > best_tm:
            best_tm = val_metrics['tm_score']
            ckpt['best_tm'] = best_tm
            torch.save(ckpt, os.path.join(cfg['training']['save_dir'], 'best_model.pt'))
            print(f'  New best TM-score: {best_tm:.4f}')

        torch.save(ckpt, os.path.join(cfg['training']['save_dir'], 'last_model.pt'))

    print(f'Training complete. Best TM-score: {best_tm:.4f}')
else:
    print('Training skipped. Set RUN_TRAINING = True to train.')

## 7. Load Model and Test Data

In [ ]:
# Load test sequences (or create sample data for local testing)
test_csv_path = CONFIG['data']['test_csv']

if os.path.exists(test_csv_path):
    test_df = pd.read_csv(test_csv_path)
    print(f'Loaded test_sequences.csv: {len(test_df)} targets')
else:
    print(f'test_sequences.csv not found at {test_csv_path}')
    print('Creating sample test data for local development...')
    sample_data = {
        'target_id': ['SAMPLE_001', 'SAMPLE_002', 'SAMPLE_003'],
        'sequence': [
            'GGGCGAUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA',
            'GGUCCGAGCAGAAGACGGCUACCCAUUCCGAUUGAGUCCUAGAAAGCUUCUUCUUUAAUUUU',
            'GCGACCGGGGCUGGCUUGGUAAUGGUACUCCCCUGUCACGGGAGAGAAUGUGGGUUCAAAUCCCAUCGGUCGCGCCA',
        ],
    }
    test_df = pd.DataFrame(sample_data)
    os.makedirs(INPUT_DIR, exist_ok=True)
    test_df.to_csv(test_csv_path, index=False)
    print(f'Sample data saved to {test_csv_path}')

print(f'Test targets: {len(test_df)}')
print(f'Sequence lengths: {test_df["sequence"].str.len().tolist()}')
print(test_df.head())

In [ ]:
# Load model for inference (dropout=0)
MODEL_CONFIG = CONFIG['model'].copy()
MODEL_CONFIG['dropout'] = 0.0
MODEL_CONFIG['max_seq_len'] = CONFIG['data']['max_seq_len']

model = RNAFoldModel(**MODEL_CONFIG).to(device)

ckpt_path = os.path.join(MODEL_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print('Loaded trained model weights.')
else:
    print('WARNING: No checkpoint found. Using random weights.')

model.eval()
print(f'Model params: {model.count_parameters():,}')

## 8. Generate Predictions

In [ ]:
@torch.no_grad()
def predict_structure(model, sequence, max_seq_len, device):
    """Predict 5 3D structures for an RNA sequence."""
    tokens = encode_sequence(sequence)
    seq_len = len(tokens)

    if seq_len <= max_seq_len:
        padded = tokens + [0] * (max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
        tokens_t = torch.tensor([padded], dtype=torch.long, device=device)
        mask_t = torch.tensor([mask], dtype=torch.bool, device=device)
        pred = model(tokens_t, mask_t)
        coords = pred['coords'][0, :seq_len].cpu().numpy()
        confidence = pred['confidence'][0, :seq_len].cpu().numpy()
    else:
        # Sliding window with overlap for long sequences
        window = max_seq_len
        stride = max_seq_len // 2
        coords = np.zeros((seq_len, model.num_predictions, 3), dtype=np.float32)
        confidence = np.zeros((seq_len, model.num_predictions), dtype=np.float32)
        weights = np.zeros((seq_len, 1, 1), dtype=np.float32)
        for start in range(0, seq_len, stride):
            end = min(start + window, seq_len)
            chunk = tokens[start:end]
            chunk_len = len(chunk)
            padded = chunk + [0] * (window - chunk_len)
            mask = [1] * chunk_len + [0] * (window - chunk_len)
            tokens_t = torch.tensor([padded], dtype=torch.long, device=device)
            mask_t = torch.tensor([mask], dtype=torch.bool, device=device)
            pred = model(tokens_t, mask_t)
            coords[start:end] += pred['coords'][0, :chunk_len].cpu().numpy()
            confidence[start:end] += pred['confidence'][0, :chunk_len].cpu().numpy()
            weights[start:end] += 1.0
            if end >= seq_len:
                break
        coords = coords / np.maximum(weights, 1e-8)
        confidence = confidence / np.maximum(weights[:, :, 0], 1e-8)

    return {'coords': coords, 'confidence': confidence}


# Generate submission
rows = []
max_seq_len = CONFIG['data']['max_seq_len']

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
    target_id = row['target_id']
    sequence = row['sequence']
    result = predict_structure(model, sequence, max_seq_len, device)
    coords = result['coords']

    for resid, nuc in enumerate(sequence):
        entry = {
            'ID': f'{target_id}_{resid + 1}',
            'resname': nuc.upper(),
            'resid': resid + 1,
        }
        for p in range(5):
            entry[f'x_{p+1}'] = round(float(coords[resid, p, 0]), 3)
            entry[f'y_{p+1}'] = round(float(coords[resid, p, 1]), 3)
            entry[f'z_{p+1}'] = round(float(coords[resid, p, 2]), 3)
        rows.append(entry)

submission = pd.DataFrame(rows)
print(f'\nSubmission shape: {submission.shape}')
print(submission.head(10))

## 9. Save Submission

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(output_path, index=False)
print(f'Submission saved to {output_path}')
print(f'Total rows: {len(submission)}')
print(f'Unique targets: {submission["ID"].str.rsplit("_", n=1).str[0].nunique()}')

# Sanity checks
coord_cols = [c for c in submission.columns if c.startswith(('x_', 'y_', 'z_'))]
print(f'\nCoordinate statistics:')
print(submission[coord_cols].describe())